In [6]:
from dataclasses import dataclass
import torch.nn as nn
import torch
from torch import Tensor
@dataclass

class PocketConfig:
	vocab_size = 50257 # tiktoken
	seq_len = 256 # context window
	d_model = 384 # embedding dimension
	n_layers = 6 # no. stacked decoder blocks
	n_heads = 6 # Query heads
	n_kv_heads = 6 # kv heads
	dropout = 0.0 # regularization

	@property
	def  d_k(self) ->  int:
		# dimension per attention
		return  self.d_model  // self.n_heads  # 64
	
	@property
	def  n_groups(self) ->  int:
		# 3 query heads share each of 2 kv heads
		return  self.n_heads  //  self.n_kv_heads # 3

	@property 
	def ffn_hidden(self) -> int:
		# ffn hidden dimension, 4x d_model rounded to chunks
		multiple = 64
		return multiple * ((4 * self.d_model + multiple - 1) // multiple - 1)// multiple

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, eps = 1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, tx: Tensor) -> Tensor:
        root_mean_square = tx.pow(2).mean(dim=-1, keepdim=True).sqrt() + self.eps

        return tx / root_mean_square

In [ ]:
class RoPE(nn.Module):
    def __init__(self, d_k, seq_len):
        super().__init__()
        theta = 1.0 / torch.pow(10000, torch.arange(0, d_k, 2).float() / d_k)
        positions = torch.arange(seq_len).float()
        
        angles = torch.outer(positions, theta)
        angles = torch.cat([angles, angles], dim=-1)

        self.register_buffer('cos', angles.cos())
        self.register_buffer('sin', angles.sin())

    def rotate_half(self, tx) -> Tensor:
        half = tx.shape[-1] // 2
        x1 = tx[..., :half]
        x2 = tx[..., half:]
        return torch.cat([-x2, x1], dim=-1)
    
    def forward(self, tx: Tensor) -> Tensor:
        seq_len = tx.shape[2]
        cos = self.cos[:seq_len, :]
        sin = self.sin[:seq_len, :]
        return tx * cos + self.rotate_half(tx) * sin
    